# 04 · Validate — linear-vs-cyclic + peptide-vs-protein modality comparison

**Standard slot:** *validate (in silico).* **For Project 09 this is the core comparison:** the two
modality benchmarks — **linear vs cyclic** (does cyclization help, at what cost?) and **peptide vs
mini-protein** (the foil) — plus **cleft-engagement reasoning vs p53** and **cyclization-feasibility
notes**, with publication-style figures (D3 part 2).

Needs `results/linear_designs.csv` + `results/macrocycle_designs.csv` +
`results/miniprotein_foil_designs.csv` + `results/all_ranked.csv` (from notebooks 02–03).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Head-to-head hit rate + interface metrics (per modality)

Compare the modalities on (a) all-layers **hit rate** and (b) the **pae_interaction** distribution of
survivors. A fair comparison filters all arms identically (notebook 03) and reports the *distribution*,
not the single best. The Boltz-2 affinity column is a **relative rank** we carry for prioritization,
never a K_D. Mock numbers are SYNTHETIC.

In [ ]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ranked = pd.read_csv("results/all_ranked.csv")
print("modalities:", ranked["modality"].value_counts().to_dict())

# Join the Boltz-2 rank + cleft_overlap from the pool CSVs onto the ranked survivors.
# (Drop any duplicate design_ids so the .map() index is unique — design_ids should already be unique.)
pools = pd.concat([pd.read_csv("results/linear_designs.csv"),
                   pd.read_csv("results/macrocycle_designs.csv"),
                   pd.read_csv("results/miniprotein_foil_designs.csv")], ignore_index=True)
pmap = pools.drop_duplicates("design_id").set_index("design_id")
ranked["boltz_affinity_score"] = ranked["design_id"].map(pmap["boltz_affinity_score"])
ranked["cleft_overlap"] = ranked["design_id"].map(pmap["cleft_overlap"])

summary = []
for m, g in ranked.groupby("modality"):
    n = len(g); passed = int((g["layers_passed"] >= 3).sum())
    summary.append(dict(modality=m, n=n, all_layers_survivors=passed,
                        hit_rate_pct=round(100*passed/max(n,1), 1),
                        median_pae=round(float(g["pae_interaction"].median()), 2),
                        median_cleft_overlap=round(float(g["cleft_overlap"].median()), 3),
                        median_boltz_rank=round(float(g["boltz_affinity_score"].median()), 4)))
summary = pd.DataFrame(summary)
print("\nmodality summary (SYNTHETIC if mock; boltz rank is RELATIVE, not a K_D):")
print(summary.to_string(index=False))

In [ ]:
# pae_interaction + Boltz-2 relative-rank distributions per modality.
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for m, g in ranked.groupby("modality"):
    ax[0].hist(g["pae_interaction"].dropna(), bins=15, alpha=0.5, label=m)
    ax[1].hist(g["boltz_affinity_score"].dropna(), bins=15, alpha=0.5, label=m)
ax[0].set_xlabel("pae_interaction (Å, lower better)"); ax[0].set_ylabel("designs"); ax[0].set_title("AF2/Boltz-2 pae_interaction"); ax[0].legend()
ax[1].set_xlabel("Boltz-2 affinity score (RELATIVE rank, NOT a K_D)"); ax[1].set_title("Boltz-2 relative ranking"); ax[1].legend()
fig.suptitle("Modality comparison (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p09_modality.png", dpi=150); plt.show()
print("saved results/p09_modality.png")

## 2 · Linear vs cyclic, explicitly

The first headline comparison. Restrict to the two **peptide** modalities (drop the mini-protein foil)
and compare hit rate, `pae_interaction`, and cleft overlap. Frame the result with the chemistry:
cyclization typically **rigidifies** the bound conformation and buys **protease stability** — at a
**synthesis cost**. Higher predicted engagement for cyclic *and* a feasible ring is the win.

In [ ]:
lin_cyc = ranked[ranked["modality"].isin(["linear", "macrocycle"])]
print("LINEAR vs CYCLIC (survivors; SYNTHETIC if mock):")
for m, g in lin_cyc.groupby("modality"):
    surv = g[g["layers_passed"] >= 3]
    print(f"  {m:11s}: n={len(g)}  hit_rate={100*len(surv)/max(len(g),1):.1f}%  "
          f"median_pae={g['pae_interaction'].median():.2f}  "
          f"median_cleft_overlap={g['cleft_overlap'].median():.2f}")
print("\nInterpretation: cyclization rigidifies + buys protease stability (validate in nb05), at a")
print("synthesis cost. Report the DISTRIBUTION + N, not the single best. Predicted affinity is unreliable.")

## 3 · Peptide vs mini-protein (the foil), explicitly

The second headline comparison: when is a **peptide/macrocycle** the right modality versus a folded
**mini-binder**? Mini-proteins usually have **higher in-silico hit rates** and easy E. coli
expression; peptides/macrocycles offer **oral / cell-penetrant** potential but are harder to get right
and need SPPS + stability/permeability validation. Report both arms honestly.

In [ ]:
pep = ranked[ranked["modality"].isin(["linear", "macrocycle"])]
foil = ranked[ranked["modality"] == "miniprotein"]
def _hit(g):
    return 100 * int((g["layers_passed"] >= 3).sum()) / max(len(g), 1)
print("PEPTIDE/MACROCYCLE vs MINI-PROTEIN FOIL (SYNTHETIC if mock):")
print(f"  peptide+macrocycle: n={len(pep)}  hit_rate={_hit(pep):.1f}%  median_pae={pep['pae_interaction'].median():.2f}")
print(f"  mini-protein foil : n={len(foil)} hit_rate={_hit(foil):.1f}%  median_pae={foil['pae_interaction'].median():.2f}")
print("\nModality trade-off: mini-proteins = higher hit rate + easy expression; peptides/macrocycles =")
print("oral/cell-penetrant potential but modest hit rate + SPPS/stability/permeability validation.")

## 4 · Cleft-engagement vs p53 + cyclization-feasibility notes `[extension]`

A peptide only **displaces p53** if it covers the p53 sub-pockets. `cleft_overlap` is our geometry
proxy: the fraction of cleft residues the peptide contacts. Higher ⇒ more likely a competitor. We also
flag a **cyclization-feasibility** note per macrocycle (a teaching heuristic on ring size; on Colab,
reason about head-to-tail vs side-chain closure and non-canonical residues for real).

In [ ]:
surv = ranked[ranked["layers_passed"] >= 3].copy()
print("cleft-engagement overlap of survivors (SYNTHETIC if mock):")
for m, g in surv.groupby("modality"):
    print(f"  {m:11s}: median p53-cleft overlap = {g['cleft_overlap'].median():.2f}  (n={len(g)})")

# Likely p53 competitors = survivors that also cover enough of the cleft.
COMPETE_OVERLAP = 0.5
competitors = surv[surv["cleft_overlap"] >= COMPETE_OVERLAP]
print(f"\nlikely p53 competitors (survivor AND cleft_overlap>={COMPETE_OVERLAP}): {len(competitors)}")
print(competitors.groupby("modality").size().to_dict())

# Cyclization-feasibility heuristic (TEACHING ONLY): flag macrocycle ring sizes outside a sane window.
macro_pool = pd.read_csv("results/macrocycle_designs.csv")
def ring_feasible(L):
    # Head-to-tail macrocycles are typically synthesizable for ~5-15 residues; outside that, flag for review.
    return 5 <= int(L) <= 15
macro_pool["ring_feasible_heuristic"] = macro_pool["length"].apply(ring_feasible)
print("\nmacrocycle ring-size feasibility (TEACHING heuristic; verify real chemistry on Colab):")
print(macro_pool["ring_feasible_heuristic"].value_counts().to_dict())

## 5 · Select the top candidates per modality

The D★ deliverable wants the **top candidates per modality**. Rank survivors by the composite score
and, as a tie-breaker, prefer higher p53-cleft overlap (and, for macrocycles, a feasible ring). Save the
shortlist for the validation plan (notebook 05). The Boltz-2 rank can prioritize *synthesis order* —
not pass/fail.

In [ ]:
top_per = []
for m, g in ranked.groupby("modality"):
    g2 = g[g["layers_passed"] >= 3].sort_values(
        ["score", "cleft_overlap"], ascending=False).head(15)
    top_per.append(g2)
top = pd.concat(top_per, ignore_index=True)
top.to_csv("results/top_candidates.csv", index=False)
print("wrote results/top_candidates.csv:", top.shape, "(top<=15 per modality)")
print(top.groupby("modality").size().to_dict())
top.head(8)[["design_id", "modality", "score", "pae_interaction", "cleft_overlap", "boltz_affinity_score"]]

## D3 (part 2) checklist
- [ ] **Linear vs cyclic**: hit rate + pae + cleft overlap, framed with the stability/synthesis trade-off (figure `results/p09_modality.png`).
- [ ] **Peptide vs mini-protein**: the foil comparison, honest about both arms' trade-offs.
- [ ] Cleft-engagement vs p53: overlap of survivors; "likely p53 competitor" count.
- [ ] Cyclization-feasibility notes (ring size; head-to-tail vs side-chain) for macrocycles.
- [ ] `results/top_candidates.csv`: top candidates per modality, ready for the validation plan.
- [ ] Honest discussion of failure modes + "predicted affinity is unreliable for short peptides".

**Next:** `05_validation_plan.ipynb` — the SPPS + protease-stability + permeability plan.